In [1]:
import json
import re

import matplotlib.pyplot as plt
import numba as nb
import numpy as np

from customs import extract_grid13
from genetic_algorithm import minimize

In [2]:
try:
    with open("data.json", "r") as file:
        json_data = json.load(file)
except FileNotFoundError:
    stack_depths = ["100-50bb", "50-30bb", "30-20bb", "20-10bb", "5bb"]
    n = 35
    json_data = {}

    for depth in stack_depths:
        json_data[depth] = []
        for i in range(1, n+1):
            idx = extract_grid13(f"charts/{depth}/{i}.png",
                                 bbox=(21, 299, 1290, 1308),
                                 raise_class=False)
            json_data[depth].append(idx.tolist())

    s = json.dumps(json_data, indent=4)
    s = re.sub(
        r'\[\n\s*([0-9,\s]+)\n\s*\]',
        lambda m: '[' + ' '.join(m.group(1).split()) + ']',
        s
    )
    with open("data.json", "w") as file:
        file.write(s)

In [3]:
stack_depths = ["100-50bb", "50-30bb", "30-20bb", "20-10bb", "5bb"]
data = np.stack([np.array(json_data[depth]) for depth in stack_depths],
                axis=0)
print(data.shape)

(5, 35, 13, 13)


In [4]:
@nb.njit
def formula(i, j, highcard, pocket, suit, diff):
    card_score = np.array([0, 0, 0, 0, 10, 9, 8, 7, 6, 5, 4, 3, 2])
    for k in range(4):
        card_score[k] = highcard[k]
    score = card_score[max(i, j)]
    if i == j:
        if 14 - i >= pocket:
            return score*2
        else:
            return score + pocket
    if min(i, j) < 4:
        score += card_score[min(i, j)] - 10
    if i < j:
        score += suit
    score -= diff[min(abs(i-j)-1, 4)]
    return score


@nb.njit
def score_chart(highcard, pocket, suit, diff):
    result = np.zeros((13, 13), dtype=np.float64)
    for i in range(13):
        for j in range(13):
            result[i, j] = formula(i, j, highcard, pocket, suit, diff)
    return result

In [5]:
npos = 35

x0 = np.array(
    [14, 13, 12, 11]*5 +
    [5, 5, 5, 5, 5] +
    [4, 4, 4, 4, 4] +
    [1, 2, 3, 4, 5] +
    [10]*npos,
    dtype=np.int32
)

In [6]:
@nb.njit
def loss_function(params):
    splits = np.array([5*4, 5, 5, 5, npos])
    idx = np.cumsum(splits)
    highcard = params[0:idx[0]].reshape(5, 4)
    pocket = params[idx[0]:idx[1]]
    suit = params[idx[1]:idx[2]]
    diff = params[idx[2]:idx[3]]
    thres = params[idx[3]:idx[4]]

    loss = 0
    for i in range(5): # stack_depth
        chart = score_chart(highcard[i], pocket[i], suit[i], diff)
        for j in range(npos):
            prediction = np.where(chart >= thres[j], 1, 0)
            answer = np.where(data[i, j] > 0, 1, 0)
            loss += np.sum(np.where(prediction != answer, 1, 0))
        loss += 0.01*np.sqrt(np.mean(chart**2))
    return loss/5/npos

In [7]:
result, f = minimize(
    loss_function,
    x0,
    generations=2000,
    population=100,
    elitism=2,
    tournament=3,
    crossover_rate=0.8,
    immigrants=10,
    smallstep=2,
    bigstep=10,
    smallp=0.3,
    bigp=0.03,
    local_search_n=100,
    local_search_p=2/x0.size,
    random_state=1,
    verbose=True
)

gen 2000: 8.197512    


In [8]:
splits = np.array([5*4, 5, 5, 5, npos])
idx = np.cumsum(splits)
highcard = result[0:idx[0]].reshape(5, 4)
pocket = result[idx[0]:idx[1]]
suit = result[idx[1]:idx[2]]
diff = result[idx[2]:idx[3]]
thres = result[idx[3]:idx[4]]

print(highcard)
print(pocket)
print(suit)
print(diff)
print(thres)

[[17 14 12 11]
 [17 14 12 11]
 [17 13 12 11]
 [17 13 12 11]
 [22 16 14 12]]
[13 11 10  8 11]
[6 6 6 5 2]
[0 1 3 3 3]
[14 13 12 11  9  7  2 18 17 15 15 13 12  5 15 15 14 13 12  5 15 14 13 12
  5 14 13 12  5 12 12  4 11  3  4]


In [9]:
highcard = [
    [17, 14, 12, 11],
    [17, 14, 12, 11],
    [17, 13, 12, 11],
    [17, 13, 12, 11],
    [22, 16, 14, 12]
]
pocket = [13, 11, 10, 9, 11]
suit = [6, 6, 6, 5, 2]
diff = [0, 1, 3, 3, 3]

scores = [score_chart(highcard[i], pocket[i], suit[i], diff)
          for i in range(len(stack_depths))]

thres = [0]*npos
err_tot = 0
for j in range(len(thres)):
    M = int(max(chart.max() for chart in scores))
    m = int(min(chart.min() for chart in scores))
    th, err = None, float('inf')
    for th_ in range(m, M+1):
        err_ = 0
        for i in range(data.shape[0]):
            prediction = np.where(scores[i] >= th_, 1, 0)
            answer = np.where(data[i, j] > 0, 1, 0)
            err_ += np.sum(np.where(prediction != answer, 1, 0))
        if err_ < err:
            th, err = th_, err_
    thres[j] = th
    err_tot += err

print(thres)
print(err_tot/5/npos)

[14, 13, 12, 11, 9, 7, 2, 18, 17, 15, 15, 14, 12, 5, 15, 15, 14, 13, 12, 5, 15, 14, 13, 12, 5, 14, 13, 12, 5, 12, 12, 4, 11, 3, 4]
8.194285714285714


In [10]:
scores = [score_chart(highcard[i], pocket[i], suit[i], diff)
          for i in range(len(stack_depths))]

for i in range(len(stack_depths)):
    print(stack_depths[i])
    for line in scores[i]:
        print(" ".join(f"{x:3.0f}" for x in line))

for depth, chart in zip(stack_depths, scores):
    plt.imshow(chart, cmap="rainbow")
    for k in range(12):
        plt.axvline(k + 0.5, c="black")
        plt.axhline(k + 0.5, c="black")
    plt.xticks(np.arange(13), list("AKQJT98765432"))
    plt.yticks(np.arange(13), list("AKQJT98765432"))
    plt.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
    plt.savefig(f"result_charts/{depth}/map.png")
    plt.clf()


100-50bb
 34  27  24  21  20  19  18  17  16  15  14  13  12
 21  28  22  20  17  16  15  14  13  12  11  10   9
 18  16  25  19  17  14  13  12  11  10   9   8   7
 15  14  13  24  17  15  12  11  10   9   8   7   6
 14  11  11  11  23  15  13  10   9   8   7   6   5
 13  10   8   9   9  22  14  12   9   8   7   6   5
 12   9   7   6   7   8  21  13  11   8   7   6   5
 11   8   6   5   4   6   7  20  12  10   7   6   5
 10   7   5   4   3   3   5   6  19  11   9   6   5
  9   6   4   3   2   2   2   4   5  18  10   8   5
  8   5   3   2   1   1   1   1   3   4  17   9   7
  7   4   2   1   0   0   0   0   0   2   3  16   8
  6   3   1   0  -1  -1  -1  -1  -1  -1   1   2  15
50-30bb
 34  27  24  21  20  19  18  17  16  15  14  13  12
 21  28  22  20  17  16  15  14  13  12  11  10   9
 18  16  24  19  17  14  13  12  11  10   9   8   7
 15  14  13  22  17  15  12  11  10   9   8   7   6
 14  11  11  11  21  15  13  10   9   8   7   6   5
 13  10   8   9   9  20  14  12   9   8   7   6

<Figure size 640x480 with 0 Axes>

In [11]:
for i, depth in enumerate(stack_depths):
    for j in range(npos):
        scores = score_chart(highcard[i], pocket[i], suit[i], diff)
        prediction = np.where(scores >= thres[j], 1, 0)
        answer = np.where(data[i, j] > 0, 1, 0)
        diff_i, diff_j = np.where(prediction != answer)
        plt.imshow(prediction, cmap="binary", vmax=2, vmin=0)
        for y, x in zip(diff_i, diff_j):
            plt.text(x, y, "X", verticalalignment="center", horizontalalignment="center", c="red", size=15)
        for k in range(12):
            plt.axvline(k + 0.5, c="black")
            plt.axhline(k + 0.5, c="black")
        plt.xticks(np.arange(13), list("AKQJT98765432"))
        plt.yticks(np.arange(13), list("AKQJT98765432"))
        plt.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
        plt.savefig(f"result_charts/{depth}/{j+1}.png")
        plt.clf()

<Figure size 640x480 with 0 Axes>